# 📖 Notebook 2: Vector Indexing Strategies

In Notebook 1, we saw that linear scan doesn't scale. Now we'll **benchmark IVFFlat vs HNSW** with real numbers and learn how to tune each index.

## Learning Objectives

By the end of this notebook, you'll understand:
- How IVFFlat and HNSW work internally
- **BAD**: Why exact KNN (no index) falls apart at scale
- **BETTER**: How to tune IVFFlat (lists, probes) for your dataset
- **BEST**: How to tune HNSW (m, ef_construction, ef_search) for optimal recall
- How to measure recall and make informed tradeoff decisions

## 🛠️ Setup

```bash
cd 03-technologies/databases/vector-databases
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import numpy as np
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "vector_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT version()")
print(f"✅ Connected: {cur.fetchone()[0][:40]}...")
conn.close()

## 📊 Generate a Large Benchmark Dataset

To see real performance differences, we need a dataset large enough that linear scan is noticeably slow. We'll generate **100,000 vectors** with 128 dimensions.

We also need a set of **ground truth** results — the TRUE nearest neighbors found by exact search. This lets us measure **recall**: what fraction of the true neighbors does our approximate index actually find?

In [ ]:
# Generate 100K random vectors for benchmarking

conn = get_conn()
cur = conn.cursor()

# Create benchmark table
cur.execute("DROP TABLE IF EXISTS bench_vectors")
cur.execute("""
    CREATE TABLE bench_vectors (
        id SERIAL PRIMARY KEY,
        embedding vector(128)
    )
""")

print("⏳ Inserting 100,000 vectors... ", end="", flush=True)
start = time.time()

batch_size = 2000
for batch in range(50):
    vectors = np.random.randn(batch_size, 128).astype(np.float32)
    values = []
    for vec in vectors:
        vec_str = '[' + ','.join(f'{v:.6f}' for v in vec) + ']'
        values.append(f"(\'{vec_str}\')")
    cur.execute(f"INSERT INTO bench_vectors (embedding) VALUES {','.join(values)}")

conn.commit()
insert_time = time.time() - start
print(f"done! ({insert_time:.1f}s)")

cur.execute("SELECT COUNT(*) FROM bench_vectors")
print(f"📊 Table has {cur.fetchone()[0]:,} vectors × 128 dimensions")
conn.close()

In [ ]:
# Generate query vectors and compute ground truth (exact KNN)

NUM_QUERIES = 20
K = 10  # find top-10 nearest neighbors

np.random.seed(42)
query_vectors = np.random.randn(NUM_QUERIES, 128).astype(np.float32)
query_strings = ['[' + ','.join(f'{v:.6f}' for v in q) + ']' for q in query_vectors]

conn = get_conn()
cur = conn.cursor()

print(f"⏳ Computing ground truth (exact KNN for {NUM_QUERIES} queries)... ", end="", flush=True)
start = time.time()

ground_truth = []
for qs in query_strings:
    cur.execute("""
        SELECT id FROM bench_vectors
        ORDER BY embedding <-> %s::vector
        LIMIT %s
    """, (qs, K))
    ground_truth.append(set(row[0] for row in cur.fetchall()))

gt_time = (time.time() - start) / NUM_QUERIES * 1000
print(f"done!")
print(f"   Average exact KNN time: {gt_time:.1f} ms per query")

conn.close()

---

## ❌ BAD: No Index (Exact KNN)

Without an index, every query scans ALL 100,000 vectors. The recall is perfect (100%), but the performance is unacceptable for any real-time application.

| Vectors | Approx Time per Query |
|---------|----------------------|
| 10,000 | ~5-15 ms |
| 100,000 | ~50-150 ms |
| 1,000,000 | ~500-1500 ms |
| 10,000,000 | ~5-15 seconds |

**This is O(n)**. Doubling your data doubles your query time.

In [ ]:
# Benchmark: No index (sequential scan)

conn = get_conn()
cur = conn.cursor()

# Make sure no vector indexes exist
cur.execute("DROP INDEX IF EXISTS idx_bench_ivfflat")
cur.execute("DROP INDEX IF EXISTS idx_bench_hnsw")
conn.commit()

print("❌ BAD: Sequential Scan (No Index)")
print("=" * 50)

times_exact = []
for qs in query_strings:
    start = time.time()
    cur.execute("""
        SELECT id FROM bench_vectors
        ORDER BY embedding <-> %s::vector
        LIMIT %s
    """, (qs, K))
    cur.fetchall()
    times_exact.append((time.time() - start) * 1000)

avg_exact = sum(times_exact) / len(times_exact)
p95_exact = sorted(times_exact)[int(len(times_exact) * 0.95)]

print(f"   Queries:         {NUM_QUERIES}")
print(f"   Avg query time:  {avg_exact:.1f} ms")
print(f"   P95 query time:  {p95_exact:.1f} ms")
print(f"   Recall:          100% (exact)")
print(f"   Vectors scanned: 100,000 (every single one)")

conn.close()

---

## ✅ BETTER: IVFFlat — Partitioned Search

### How IVFFlat Works (Under the Hood)

**Build Phase:**
1. Run K-means clustering to find `lists` centroids
2. Assign each vector to its nearest centroid
3. Store vectors grouped by their cluster

**Query Phase:**
1. Find the `probes` nearest centroids to the query vector
2. Only scan vectors in those clusters
3. Return the top-K results from the scanned vectors

```
              Build: K-means creates clusters
    ┌──────┐  ┌──────┐  ┌──────┐  ┌──────┐
    │  C1  │  │  C2  │  │  C3  │  │  C4  │   (centroids)
    │ •••  │  │ •••  │  │ •••  │  │ •••  │   (vectors in each cluster)
    │ •••  │  │ ••   │  │ •••  │  │ ••   │
    └──────┘  └──────┘  └──────┘  └──────┘

    Query: probes = 2 → search C1 and C3 (nearest centroids)
    Skip C2 and C4 entirely!
```

### Tuning Parameters

| Parameter | What It Controls | Rule of Thumb |
|-----------|-----------------|---------------|
| `lists` | Number of clusters | `sqrt(n)` to `n/1000` |
| `probes` | Clusters searched per query | Start at `sqrt(lists)`, increase for recall |

In [ ]:
# BETTER: Build IVFFlat index with sqrt(n) lists

conn = get_conn()
cur = conn.cursor()

nlists = 316  # sqrt(100000) ≈ 316

print(f"⏳ Building IVFFlat (lists={nlists})... ", end="", flush=True)
start = time.time()
cur.execute(f"""
    CREATE INDEX idx_bench_ivfflat
    ON bench_vectors
    USING ivfflat (embedding vector_l2_ops)
    WITH (lists = {nlists})
""")
conn.commit()
ivf_build_time = time.time() - start
print(f"done! ({ivf_build_time:.1f}s)")

conn.close()

In [ ]:
# Benchmark IVFFlat with different probes settings

conn = get_conn()
cur = conn.cursor()

print("✅ BETTER: IVFFlat Benchmark (lists=316)")
print("=" * 65)
print(f"{'Probes':>8} {'Avg Time (ms)':>15} {'P95 Time (ms)':>15} {'Recall':>8}")
print("-" * 65)

probes_options = [1, 5, 10, 20, 50]

ivf_results = []
for nprobes in probes_options:
    cur.execute(f"SET ivfflat.probes = {nprobes}")

    times = []
    recall_scores = []

    for i, qs in enumerate(query_strings):
        start = time.time()
        cur.execute("""
            SELECT id FROM bench_vectors
            ORDER BY embedding <-> %s::vector
            LIMIT %s
        """, (qs, K))
        results = set(row[0] for row in cur.fetchall())
        times.append((time.time() - start) * 1000)

        # Compare with ground truth
        recall = len(results & ground_truth[i]) / K
        recall_scores.append(recall)

    avg_t = sum(times) / len(times)
    p95_t = sorted(times)[int(len(times) * 0.95)]
    avg_recall = sum(recall_scores) / len(recall_scores)

    print(f"{nprobes:>8} {avg_t:>15.1f} {p95_t:>15.1f} {avg_recall:>7.1%}")
    ivf_results.append((nprobes, avg_t, avg_recall))

conn.close()

print()
print("💡 Key Insight: More probes = better recall but slower queries.")
print("   probes=10 is often a good starting point (90%+ recall, fast queries).")

---

## 🏆 BEST: HNSW — Graph-Based Search

### How HNSW Works (Under the Hood)

**Build Phase:**
1. Create a multi-layer graph (like a skip list)
2. Each vector is a node, connected to its `m` nearest neighbors
3. Higher layers have fewer nodes (coarse navigation)
4. Lower layers have more nodes (fine navigation)

**Query Phase:**
1. Enter at the top layer
2. Greedily move to the nearest node at each layer
3. Drop down to the next layer and repeat
4. At the bottom layer, explore `ef_search` candidates
5. Return the top-K results

```
Layer 2:  A ───────────── D ───────────── G
          │               │               │
Layer 1:  A ─── C ─────── D ─── F ─────── G
          │     │         │     │         │
Layer 0:  A─B─C─D─E─F─G─H─I─J─K─L─M─N─O─P

Query: Start at A(layer 2) → jump to D → drop to layer 1
       → move to F → drop to layer 0 → explore neighbors → found!
```

### Tuning Parameters

| Parameter | What It Controls | Default | Tradeoff |
|-----------|-----------------|---------|----------|
| `m` | Max connections per node | 16 | Higher = better recall, more memory |
| `ef_construction` | Build-time search width | 64 | Higher = better index, slower build |
| `ef_search` | Query-time search width | 40 | Higher = better recall, slower query |

In [ ]:
# BEST: Build HNSW index

conn = get_conn()
cur = conn.cursor()

cur.execute("DROP INDEX IF EXISTS idx_bench_ivfflat")
cur.execute("DROP INDEX IF EXISTS idx_bench_hnsw")
conn.commit()

print("⏳ Building HNSW (m=16, ef_construction=64)... ", end="", flush=True)
start = time.time()
cur.execute("""
    CREATE INDEX idx_bench_hnsw
    ON bench_vectors
    USING hnsw (embedding vector_l2_ops)
    WITH (m = 16, ef_construction = 64)
""")
conn.commit()
hnsw_build_time = time.time() - start
print(f"done! ({hnsw_build_time:.1f}s)")

conn.close()

In [ ]:
# Benchmark HNSW with different ef_search settings

conn = get_conn()
cur = conn.cursor()

print("🏆 BEST: HNSW Benchmark (m=16, ef_construction=64)")
print("=" * 65)
print(f"{'ef_search':>10} {'Avg Time (ms)':>15} {'P95 Time (ms)':>15} {'Recall':>8}")
print("-" * 65)

ef_options = [10, 20, 40, 100, 200]

hnsw_results = []
for ef in ef_options:
    cur.execute(f"SET hnsw.ef_search = {ef}")

    times = []
    recall_scores = []

    for i, qs in enumerate(query_strings):
        start = time.time()
        cur.execute("""
            SELECT id FROM bench_vectors
            ORDER BY embedding <-> %s::vector
            LIMIT %s
        """, (qs, K))
        results = set(row[0] for row in cur.fetchall())
        times.append((time.time() - start) * 1000)

        recall = len(results & ground_truth[i]) / K
        recall_scores.append(recall)

    avg_t = sum(times) / len(times)
    p95_t = sorted(times)[int(len(times) * 0.95)]
    avg_recall = sum(recall_scores) / len(recall_scores)

    print(f"{ef:>10} {avg_t:>15.1f} {p95_t:>15.1f} {avg_recall:>7.1%}")
    hnsw_results.append((ef, avg_t, avg_recall))

conn.close()

print()
print("💡 Key Insight: ef_search=40-100 typically gives 95%+ recall with fast queries.")
print("   HNSW generally achieves higher recall than IVFFlat at the same speed.")

---

## 📊 Head-to-Head Comparison

In [ ]:
# Final comparison: No Index vs IVFFlat vs HNSW

print("📊 Head-to-Head: IVFFlat vs HNSW (100,000 vectors)")
print("=" * 75)
print()

# Use the middle benchmark results for comparison
ivf_mid = ivf_results[2]  # probes=10
hnsw_mid = hnsw_results[2]  # ef_search=40

print(f"{'Metric':<25} {'No Index':>14} {'IVFFlat':>14} {'HNSW':>14}")
print("-" * 75)
print(f"{'Query time':<25} {f'{avg_exact:.1f} ms':>14} {f'{ivf_mid[1]:.1f} ms':>14} {f'{hnsw_mid[1]:.1f} ms':>14}")
print(f"{'Recall':<25} {'100%':>14} {f'{ivf_mid[2]:.1%}':>14} {f'{hnsw_mid[2]:.1%}':>14}")
print(f"{'Build time':<25} {'N/A':>14} {f'{ivf_build_time:.1f}s':>14} {f'{hnsw_build_time:.1f}s':>14}")
print(f"{'Memory overhead':<25} {'None':>14} {'Low':>14} {'High (~2x)':>14}")
print(f"{'Best for':<25} {'<10K rows':>14} {'Quick setup':>14} {'Best recall':>14}")
print()
print("Guidelines:")
print("  < 10K vectors    → No index needed, linear scan is fine")
print("  10K - 1M vectors → HNSW is usually the best choice")
print("  > 1M vectors     → HNSW if memory allows, IVFFlat if memory-constrained")
print("  Need fast builds → IVFFlat (builds in seconds vs minutes for HNSW)")

## 🎯 Choosing Parameters: Quick Reference

### IVFFlat
```
lists  = sqrt(n)        # number of clusters
probes = sqrt(lists)    # clusters to search (start here, increase for recall)
```

### HNSW
```
m               = 16    # connections per node (16 is good default)
ef_construction = 64    # build quality (64–200, higher = better but slower build)
ef_search       = 40    # query quality (40–200, higher = better recall)
```

### When to pick which?

| Situation | Choice |
|-----------|--------|
| Need highest recall | HNSW with high ef_search |
| Memory is tight | IVFFlat (lower memory overhead) |
| Data changes frequently | IVFFlat (faster to rebuild) |
| Mostly static data | HNSW (build once, query fast) |
| Quick prototype | IVFFlat (faster to set up) |
| Production similarity search | HNSW |

## 🧹 Cleanup

In [ ]:
conn = get_conn()
cur = conn.cursor()
cur.execute("DROP TABLE IF EXISTS bench_vectors")
conn.commit()
conn.close()
print("🧹 Cleaned up benchmark table")

## 📚 Summary

### Key Takeaways

1. **Exact KNN** scans every vector — O(n), doesn't scale past ~10K vectors
2. **IVFFlat** partitions vectors into clusters — fast build, decent recall (~90-95%)
3. **HNSW** builds a navigable graph — slower build, best recall (~95-99%)
4. **Recall** measures what fraction of true nearest neighbors the index finds
5. Both indexes have tunable parameters that trade speed for recall

### Interview Tip

> "I'd use HNSW for most vector search use cases — it gives the best recall-to-speed ratio. IVFFlat is a good fallback when memory is tight or data changes frequently. The key parameters to tune are ef_search for HNSW (controls recall vs speed) and probes for IVFFlat."

### Next Up

In **Notebook 3**, we'll tackle **hybrid search** — combining vector similarity with traditional SQL filters like price, category, and availability.